# Tabular Classification — Census Income

A supervised-learning pipeline that predicts a person's income bracket from census data. Three models are compared — **Random Forest, Gradient Boosting and Logistic Regression** — with proper preprocessing pipelines and hyperparameter tuning.

**Techniques:** data cleaning, feature engineering, `ColumnTransformer` preprocessing (scaling + one-hot encoding), model comparison, `GridSearchCV` hyperparameter tuning, test-set evaluation.

### 1. Imports and data loading

In [ ]:
#Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import zipfile

with zipfile.ZipFile('data1.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/data')

#Find the file CSV
from pathlib import Path
csv_path = next(Path('/content/data').glob('*.csv'))
dataset = pd.read_csv(csv_path)

### 2. Cleaning and feature engineering
We drop redundant columns, group the many occupation values into a handful of broader categories, and remove rows with missing values.

In [ ]:
# Clean data by removing unnecessary columns
columns_to_drop = ['education', 'native-country']
existing_columns = dataset.columns

# Filter out columns that don't exist in the dataset
columns_to_drop = [col for col in columns_to_drop if col in existing_columns]

# Drop the existing columns
dataset.drop(columns_to_drop, axis=1, inplace=True)


# Group occupations into fewer categories
occupation_categories = {
    'Tech': ['Tech-support', 'Craft-repair', 'Machine-op-inspct', 'Transport-moving'],
    'Sales': ['Sales', 'Adm-clerical'],
    'Exec': ['Exec-managerial', 'Prof-specialty'],
    'Service': ['Handlers-cleaners', 'Protective-serv', 'Priv-house-serv'],
    'Other': ['Other-service', 'Farming-fishing', 'Armed-Forces']
}

def categorize_occupation(occupation):
    for group, jobs in occupation_categories.items():
        if occupation in jobs:
            return group
    return 'Other'

dataset['job_category'] = dataset['occupation'].apply(categorize_occupation)
dataset.replace('?', np.nan, inplace=True)
dataset.dropna(inplace=True)

### 3. Preprocessing and model comparison
Categorical variables are encoded and numerical ones scaled inside a single pipeline, so the same transformations are applied consistently to train and test. We then compare the three classifiers.

In [ ]:
# Encode categorical variables
categorical_columns = ['workclass', 'marital-status', 'job_category', 'relationship', 'race', 'sex']
encoder = LabelEncoder()
for col in categorical_columns:
    dataset[col] = encoder.fit_transform(dataset[col])

# Split dataset into features and target variable
features = dataset.drop('target', axis=1)
target = dataset['target']

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.3, random_state=123)

# Identify numerical and categorical columns
numerical_columns = features.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_columns = features.select_dtypes(include='object').columns.tolist()

# Preprocessing pipeline: scaling numerical features and encoding categorical features
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_columns),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
])

# Define model dictionary
models_to_evaluate = {
    'RandomForest': RandomForestClassifier(random_state=123),
    'GradientBoosting': GradientBoostingClassifier(random_state=123),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=123)
}

# Evaluate models
for model_name, model in models_to_evaluate.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print(f"Model: {model_name}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
    print(f"Classification Report:\n{classification_report(y_test, y_pred)}")
    print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
    print("\n")

Model: RandomForest
Accuracy: 0.8599651960282526
Classification Report:
              precision    recall  f1-score   support

       <=50K       0.89      0.93      0.91      7408
        >50K       0.75      0.63      0.69      2361

    accuracy                           0.86      9769
   macro avg       0.82      0.78      0.80      9769
weighted avg       0.85      0.86      0.86      9769

Confusion Matrix:
[[6910  498]
 [ 870 1491]]


Model: GradientBoosting
Accuracy: 0.8638550516941345
Classification Report:
              precision    recall  f1-score   support

       <=50K       0.88      0.95      0.91      7408
        >50K       0.79      0.59      0.68      2361

    accuracy                           0.86      9769
   macro avg       0.84      0.77      0.80      9769
weighted avg       0.86      0.86      0.86      9769

Confusion Matrix:
[[7040  368]
 [ 962 1399]]


Model: LogisticRegression
Accuracy: 0.8342716757088751
Classification Report:
              precision   

### 4. Hyperparameter tuning
We tune Random Forest, Gradient Boosting and Logistic Regression with `GridSearchCV` and 3-fold cross-validation, optimising for accuracy.

In [ ]:
# Hyperparameter tuning for Random Forest

rf_param_grid = {
    'classifier__n_estimators': [200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=123))
])

rf_grid_search = GridSearchCV(rf_model, rf_param_grid, cv=3, n_jobs=-1, scoring='accuracy')
rf_grid_search.fit(X_train, y_train)

print("Best Random Forest Params:", rf_grid_search.best_params_)
print("Best Random Forest Score:", rf_grid_search.best_score_)



# Hyperparameter tuning for Gradient Boosting
gb_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.05, 0.1, 0.2],
    'classifier__max_depth': [3, 5]
}

gb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=123))
])

gb_grid_search = GridSearchCV(gb_model, gb_param_grid, cv=3, n_jobs=-1, scoring='accuracy')
gb_grid_search.fit(X_train, y_train)

print("Best Gradient Boosting Params:", gb_grid_search.best_params_)
print("Best Gradient Boosting Score:", gb_grid_search.best_score_)

# Hyperparameter tuning for Logistic Regression
lr_param_grid = {
    'classifier__C': [0.1, 1, 10],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['lbfgs']
}

lr_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=123))
])

lr_grid_search = GridSearchCV(lr_model, lr_param_grid, cv=3, n_jobs=-1, scoring='accuracy')
lr_grid_search.fit(X_train, y_train)

print("Best Logistic Regression Params:", lr_grid_search.best_params_)
print("Best Logistic Regression Score:", lr_grid_search.best_score_)


Best Random Forest Params: {'classifier__max_depth': 20, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
Best Random Forest Score: 0.8632413317952486
Best Gradient Boosting Params: {'classifier__learning_rate': 0.2, 'classifier__max_depth': 3, 'classifier__n_estimators': 200}
Best Gradient Boosting Score: 0.8715337786424123
Best Logistic Regression Params: {'classifier__C': 1, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
Best Logistic Regression Score: 0.8308179321880579


### 5. Final evaluation on the test set
The best version of each model is evaluated on the held-out test data.

In [ ]:
# Final model evaluation on test data
rf_best_model = rf_grid_search.best_estimator_
rf_test_preds = rf_best_model.predict(X_test)
rf_test_accuracy = accuracy_score(y_test, rf_test_preds)

gb_best_model = gb_grid_search.best_estimator_
gb_test_preds = gb_best_model.predict(X_test)
gb_test_accuracy = accuracy_score(y_test, gb_test_preds)

lr_best_model = lr_grid_search.best_estimator_
lr_test_preds = lr_best_model.predict(X_test)
lr_test_accuracy = accuracy_score(y_test, lr_test_preds)

print("Random Forest Test Accuracy:", rf_test_accuracy)
print("Gradient Boosting Test Accuracy:", gb_test_accuracy)
print("Logistic Regression Test Accuracy:", lr_test_accuracy)


Random Forest Test Accuracy: 0.8662094380182209
Gradient Boosting Test Accuracy: 0.8723513153854028
Logistic Regression Test Accuracy: 0.8342716757088751
